# Stage B - EAST's architecture knobs

**Question: do DyReLU, its phasing, and weight sharing do anything?**

| Rung | Change |
|---|---|
| B1 | ReLU replaced by DyReLU |
| B2 | DyReLU phased out over training instead of held on |
| B3 | Residual blocks share weights |

Designed so **a null is publishable**: a pre-registered +/-1.0pp equivalence bound
with TOST reported alongside the difference CI. "We can reject differences larger
than 1.5pp" is a result; "p = 0.43" is not.

B3 rewrites the sparsity target against the unique parameter count, which is why
`sparsity_requested` and `sparsity_target_adjusted` are separate fields on the record.

> Paired: same substrate and the same mask-trajectory family, so the intervention is architectural, seeds correlate, and pairing pays.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))          # so ladder_nb is importable
import ladder_nb as nb
info = nb.setup()


## Configure

`TIER` 1 is the spine (5 seeds). Cells per GPU and dataloader workers are constants at the top of `pool.py`.


In [ ]:
TIER    = 1
GPUS    = info['gpus'] or 1
SEEDS   = None          # None = every seed the tier schedules

RUNGS = ['B1', 'B2', 'B3']

import manifest as M
grid = [c for c in M.cells(TIER, rungs=RUNGS)]
print(f'{len(grid)} cell(s) planned over {len(set(c["rung"] for c in grid))} rung(s)')
for r in RUNGS:
    n = sum(1 for c in grid if c['rung'] == r)
    print(f'  {r:10s} {n} seed(s)' if n else f'  {r:10s} -- NOT IN TIER {TIER}')


## Run

Idempotent - a cell is complete iff a record carrying its key exists, so re-running skips what is done. Dense cells run first as a hard barrier. Safe to interrupt; you lose at most the cells in flight.


In [ ]:
summary = nb.run_stage(RUNGS, tier=TIER, gpus=GPUS, seeds=SEEDS)
print(summary['ok'], 'ok,', summary['failed'], 'failed,', summary['skipped'], 'skipped')


## Progress and health

The `nan` column is the one to read first. A diverged run sits at exactly 10.0969% (chance on CIFAR-10) for the rest of training and every delta computed from it is meaningless.


In [ ]:
nb.progress(TIER)


## Watch a single cell

Use this when something looks wrong - it streams one line per epoch so you can see *where* a run breaks rather than only that it did.


In [ ]:
cell = nb.attach(nb.pick('B1', seed=1, tier=TIER))
nb.show(cell)
# hist, first_nan = nb.watch(cell, gpu=0)
# nb.plot(hist, first_nan, cell['key'])


## Table and gates


In [ ]:
out = nb.report(TIER)
